In [ ]:
!mkdir -p week-2
%cd week-2
!pwd

/content/week-2
/content/week-2


In [ ]:
!apt-get -qq install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

Selecting previously unselected package zstd.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
!ollama --version

In [14]:
import subprocess, time, requests

ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

for _ in range(30):
    try:
        r = requests.get("http://localhost:11434")
        if r.ok:
            print("Server says:", r.text)
            break
    except requests.exceptions.ConnectionError:
        time.sleep(1)
else:
    print("Server did not start. Re-run this cell.")

Server says: Ollama is running


In [ ]:
!ollama pull llama3.2


Error: 503: 


In [ ]:
import requests

r = requests.get("http://localhost:11434")
print(r.status_code)
print(r.text)

200
Ollama is running


In [16]:
!ollama list

NAME               ID              SIZE      MODIFIED      
llama3.2:latest    a80c4f17acd5    2.0 GB    2 seconds ago    


In [15]:
!ollama pull llama3.2

In [17]:
!curl -s http://localhost:11434/api/generate -d '{"model": "llama3.2", "prompt": "Say hello in one short sentence.", "stream": false}'

{"model":"llama3.2","created_at":"2026-11-19T04:58:11.538055576Z","response":"Hello!","done":true,"done_reason":"stop","context":[128006,9125,128007,271,38766,1303,33025,2696,25,6790,220,2366,18,271,128009,128006,882,128007,271,46864,24748,304,832,2875,11914,13,128009,128006,78191,128007,271,9906,0],"total_duration":23229895515,"load_duration":3586514718,"prompt_eval_count":32,"prompt_eval_cached_count":0,"prompt_eval_duration":19601029000,"eval_count":3,"eval_duration":38790000}

In [18]:
from google.colab import userdata

key = userdata.get("GEMINI_API_KEY")

with open(".env", "w") as f:
    f.write(f'GEMINI_API_KEY="{key}"\n')

print(".env created. Key length:", len(key))

.env created. Key length: 53


In [19]:
%%writefile requirements.txt
python-dotenv
pytest
google-genai
requests

Writing requirements.txt


In [20]:
!pip install -q -r requirements.txt

In [21]:
from google import genai
print("google-genai imported OK")

google-genai imported OK


In [22]:

%%writefile llm_client.py
from abc import ABC, abstractmethod
from google import genai
import requests


# Common interface
class LLMProvider(ABC):

    @abstractmethod
    def generate(self, prompt):
        pass


# Gemini cloud provider
class GeminiProvider(LLMProvider):

    def __init__(self, api_key, model="gemini-3.8-flash"):
        self.client = genai.Client(api_key=api_key)
        self.model = model

    def generate(self, prompt):
        response = self.client.models.generate_content(
            model=self.model,
            contents=prompt
        )
        return response.text


# Ollama local provider
class OllamaProvider(LLMProvider):

    def __init__(self, model="llama3.2"):
        self.model = model
        self.url = "http://localhost:11434/api/generate"

    def generate(self, prompt):
        response = requests.post(
            self.url,
            json={
                "model": self.model,
                "prompt": prompt,
                "stream": False
            },
            timeout=120
        )
        response.raise_for_status()
        return response.json()["response"]


# Common client
class LLMClient:

    def __init__(self, provider):
        self.provider = provider

    def generate(self, prompt):
        return self.provider.generate(prompt)


Writing llm_client.py


In [23]:
!ls

llm_client.py  requirements.txt


In [30]:

%%writefile main.py
import os
from dotenv import load_dotenv
from llm_client import (
    GeminiProvider,
    OllamaProvider,
    LLMClient
)

load_dotenv(override=True)

provider_name = input(
    "Choose provider (gemini/ollama): "
).strip().lower()

if provider_name == "gemini":
    api_key = os.getenv("GEMINI_API_KEY")
    if not api_key:
        raise ValueError("Gemini API key not found in .env")
    provider = GeminiProvider(api_key)

elif provider_name == "ollama":
    provider = OllamaProvider("llama3.2")

else:
    raise ValueError("Invalid provider selected")

client = LLMClient(provider)

prompt = input("Enter your question: ")
answer = client.generate(prompt)

print("\nLLM Response:")
print(answer)


Overwriting main.py


In [31]:

%%writefile test_client.py
import unittest
from llm_client import LLMClient


# Fake provider for testing
class FakeProvider:
    def generate(self, prompt):
        return "AI stands for Artificial Intelligence."


class TestLLMClient(unittest.TestCase):

    def test_generate(self):
        provider = FakeProvider()
        client = LLMClient(provider)

        result = client.generate("Explain AI")

        self.assertEqual(
            result,
            "AI stands for Artificial Intelligence."
        )


if __name__ == "__main__":
    unittest.main()


Overwriting test_client.py


In [32]:
!python -m pytest test_client.py -v

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/week-2
plugins: typeguard-4.6.0, platformdirs-4.14.3, langsmith-0.14.17, anyio-4.16.0
collected 1 item                                                               

test_client.py::TestLLMClient::test_generate PASSED                      [100%]

============================== 1 passed in 1.22s ===============================


In [33]:
%run main.py

Choose provider (gemini/ollama): ollama
Enter your question: What is GenAI

LLM Response:
GenAI, short for General Artificial Intelligence, refers to a hypothetical AI system that possesses the ability to understand, learn, and apply knowledge across a wide range of tasks, similar to human intelligence. GenAI systems aim to mimic human-like intelligence, reasoning, and problem-solving capabilities, making them potentially capable of performing any intellectual task that humans can.

The concept of GenAI is often referred to as the "Holy Grail" of artificial intelligence, as it represents the ultimate goal of creating a machine that can think, learn, and reason like a human being. While significant progress has been made in AI research, GenAI remains a subject of ongoing debate and research in the field.

Some of the key characteristics of GenAI include:

1. **Generalizability**: GenAI systems should be able to apply their knowledge and skills to a wide range of tasks and domains, witho

In [34]:
%run main.py

Choose provider (gemini/ollama): gemini
Enter your question: What's difference between Machine Learning and Deep learning


ClientError: 403 PERMISSION_DENIED. {'error': {'code': 403, 'message': 'The Gemini API free tier is not available for this project. To use the Gemini API, please set up billing to upgrade to the paid tier at https://aistudio.google.com/billing. For more information, see https://ai.google.dev/gemini-api/docs/troubleshooting#project-unavailable.', 'status': 'PERMISSION_DENIED'}}

In [29]:
!ls -la

total 36
drwxr-xr-x 4 root root 4096 Nov 19 05:03 .
drwxr-xr-x 1 root root 4096 Nov 19 04:48 ..
-rw-r--r-- 1 root root   71 Nov 19 05:00 .env
-rw-r--r-- 1 root root 1283 Nov 19 05:01 llm_client.py
-rw-r--r-- 1 root root  701 Nov 19 05:02 main.py
drwxr-xr-x 2 root root 4096 Nov 19 05:03 __pycache__
drwxr-xr-x 3 root root 4096 Nov 19 05:03 .pytest_cache
-rw-r--r-- 1 root root   43 Nov 19 05:00 requirements.txt
-rw-r--r-- 1 root root  537 Nov 19 05:02 test_client.py
